### 02.- Ingest Core Data
Ingesta incremental de datos maestros (usuarios y tarjetas) desde la capa RAW hacia Bronze.


In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container",       "raw")
dbutils.widgets.text("containerDestino","bronze")
dbutils.widgets.text("catalogo",        "bank_dev")
dbutils.widgets.text("esquema",         "bronze")
dbutils.widgets.text("storageName",     "saprojectbankdeveastus")

In [0]:
container        = dbutils.widgets.get("container")
containerDestino = dbutils.widgets.get("containerDestino")
catalogo         = dbutils.widgets.get("catalogo")
esquema          = dbutils.widgets.get("esquema")
storageName      = dbutils.widgets.get("storageName")

In [0]:
storage_path         = f"abfss://{container}@{storageName}.dfs.core.windows.net/"
storage_path_destino = f"abfss://{containerDestino}@{storageName}.dfs.core.windows.net/"

# Fuentes (RAW)
raw_users_root = f"{storage_path}core_banco/users_data/"
raw_cards_root = f"{storage_path}core_banco/cards_data/"

# Checkpoints y Schema Locations (BRONZE)
schema_location_users     = f"{storage_path_destino}_schemas/users"
checkpoint_location_users = f"{storage_path_destino}_checkpoints/users"

schema_location_cards     = f"{storage_path_destino}_schemas/cards"
checkpoint_location_cards = f"{storage_path_destino}_checkpoints/cards"

In [0]:
from pyspark.sql import functions as F

## Users

In [0]:
# Parquet tiene schema embebido — NO se provee .schema() explícito.
# Auto Loader infiere los tipos directamente del archivo (BIGINT, DOUBLE, INT, etc.).
# cloudFiles.schemaLocation rastrea la evolución del schema entre ejecuciones.
df_stream_users = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_location_users)
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(raw_users_root)
)

df_bronze_users = df_stream_users.withColumn("ingestion_date", F.current_timestamp())

query_users = (
    df_bronze_users.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location_users)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"{catalogo}.{esquema}.users")
)

## Cards

In [0]:
# card_number y cvv son campos PII — quedan como STRING en el Parquet fuente.
# Parquet tiene schema embebido — NO se provee .schema() explícito.
df_stream_cards = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_location_cards)
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(raw_cards_root)
)

df_bronze_cards = df_stream_cards.withColumn("ingestion_date", F.current_timestamp())

query_cards = (
    df_bronze_cards.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location_cards)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"{catalogo}.{esquema}.cards")
)